# Build USL-Suspilne Dataset

End-to-end pipeline: Firebase → annotations → download videos → split → crop signer → identify signers → write dataset splits → extract poses.

**Output:** `data/usl-suspilne/` (publishable) — `{train,dev,test,test_unseen}.csv`, `features/{videoId}/{clip}.mp4`, `poses/mediapipe_holistic/{videoId}/{clip}.npy`. Intermediates (`annotations.csv`, `splits.csv`, `signers.json`, previews) live in `data/cache/`.

**Requirements:** `pip install firebase-admin yt-dlp num2words pymorphy3 pymorphy3-dicts-uk mediapipe opencv-python insightface onnxruntime scikit-learn`, `ffmpeg` installed, `serviceAccountKey.json` in repo root.

In [3]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

# Cache = working tree (mutable, regenerable, keyed by YouTube ID)
CACHE_DIR = ROOT / "data/cache"
FIREBASE_DIR = ROOT / "data/firebase"
ANNOTATIONS_CSV = CACHE_DIR / "annotations.csv"
SPLITS_CSV = CACHE_DIR / "splits.csv"
RAW_VIDEOS_DIR = CACHE_DIR / "raw_videos"
UNCROPPED_DIR = CACHE_DIR / "uncropped"
SIGNERS_JSON = CACHE_DIR / "signers.json"
SIGNER_PREVIEWS = CACHE_DIR / "signer_previews"
VIDEOS_JSON = CACHE_DIR / "videos.json"

# Pose / feature working dirs live under cache so the build pipeline doesn't
# touch the published artefact. The publish step (§10) copies them to
# DATASET_DIR with sequential v-IDs.
FEATURES_DIR = CACHE_DIR / "features"
HOLISTIC_DIR = CACHE_DIR / "poses/mediapipe_holistic"
MP3D_DIR = CACHE_DIR / "poses/mediapipe_3d"

# Dataset = published artefact (immutable layout, v-IDs in `name`)
DATASET_DIR = ROOT / "data/usl-suspilne"

print(f"Project root: {ROOT}")


Project root: /Users/xandro/code/diploma/ucu-text-to-sign


## 1. Download Firebase Export

In [17]:
from download_firebase import download_and_save

export_path = download_and_save(output_dir=FIREBASE_DIR)
print(f"\nExport saved to: {export_path}")

Saved to /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-28.json
Updated /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/latest.json -> 2026-04-28.json
  Videos: 26 (11 marked complete)
  Video captions entries: 26
  Total captions: 6801
  Aligned captions: 5163

Export saved to: /Users/xandro/code/diploma/ucu-text-to-sign/data/firebase/2026-04-28.json


## 2. Build Annotations

Reads the Firebase export (data/firebase/latest.json), finds all captions marked as `aligned`, and writes:
- `data/cache/annotations.csv` — `name|text|text_norm|annotator`
- `data/cache/splits.csv` — `name|video|start|end|split` (clip manifest with split assignment)

`text_norm` is the normalized text: lowercased, punctuation stripped, numbers converted to words.
Filters out clips shorter than 0.3s and any excluded video IDs.

The publishable per-split CSVs (`train/dev/test/test_unseen.csv`) are written later by step 7, after signer identification.

In [5]:
from build_annotations_from_firebase import build_annotations

result = build_annotations(
    export_path=FIREBASE_DIR / "latest.json",
    output_csv=ANNOTATIONS_CSV,
    splits_csv=SPLITS_CSV,
    split_strategy="random",
)
print(f"\n{result['rows']} annotations from {len(result['videos'])} videos")


Videos: 11 complete, 15 incomplete
  split_strategy=random
  train: 1691 clips
  dev: 211 clips
  test: 212 clips
  test_unseen: 0 clips
Wrote 2114 annotations from 11 videos to /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/annotations.csv
Skipped 15 incomplete videos (use --all to include)
  2Nnz697BVTw: 85 clips
  A2hCZVvtUSE: 115 clips
  IOflFDS2biE: 481 clips
  K9ouFMtz-s8: 61 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 118 clips
  Nyykyn4FpNo: 215 clips
  Q3yRVXmZdGQ: 145 clips
  cNT6ajjEwVU: 178 clips
  jj5jiyl2mh0: 423 clips
  uGMgleLkjho: 147 clips

2114 annotations from 11 videos


## 3. Download Videos

Downloads YouTube videos at best available quality.
Skips already-downloaded videos. Use `--force VIDEO_ID` to re-download.

In [6]:
from download_videos import download_all

dl_result = download_all(splits_csv=SPLITS_CSV, dst=RAW_VIDEOS_DIR)

if dl_result["failed"]:
    print(f"\nFailed videos will be excluded in the verify step.")

Found 11 videos in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/splits.csv

[2Nnz697BVTw]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/2Nnz697BVTw.mp4 already exists
[A2hCZVvtUSE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/A2hCZVvtUSE.mp4 already exists
[IOflFDS2biE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/IOflFDS2biE.mp4 already exists
[K9ouFMtz-s8]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/K9ouFMtz-s8.mp4 already exists
[KUDt_SKkPUE]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/KUDt_SKkPUE.mp4 already exists
[MSqpwfErl34]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/MSqpwfErl34.mp4 already exists
[Nyykyn4FpNo]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/raw_videos/Nyykyn4FpNo.mp4 already exists
[Q3yRVXmZdGQ]
  [skip] /Users/xandro/code/diploma/ucu-text-to-sign/data/ca

## 4. Split into Clips

Splits videos into sentence-level clips based on timestamps.
Skips already-split clips. Use `--force VIDEO_ID` to re-split.

In [7]:
from split_videos import split_all

split_result = split_all(splits_csv=SPLITS_CSV, raw_dir=RAW_VIDEOS_DIR, dst=UNCROPPED_DIR)

Found 2114 annotations across 11 videos

[2Nnz697BVTw] 85 clips
[A2hCZVvtUSE] 115 clips
[IOflFDS2biE] 481 clips
[K9ouFMtz-s8] 61 clips
[KUDt_SKkPUE] 146 clips
[MSqpwfErl34] 118 clips
[Nyykyn4FpNo] 215 clips
[Q3yRVXmZdGQ] 145 clips
[cNT6ajjEwVU] 178 clips
[jj5jiyl2mh0] 423 clips
[uGMgleLkjho] 147 clips

Done. 2114 clips, 0 failed, 0 videos skipped.


## 5. Crop to Signer Region

Crops the bottom-right 510x510 region where the sign language interpreter appears.

In [8]:
from crop_signer import crop_all

crop_result = crop_all(src=UNCROPPED_DIR, dst=FEATURES_DIR)

Found 2115 clips in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/uncropped

Done. 2115 cropped, 0 failed.


## 6. Identify Signers (Face Clustering)

Runs RetinaFace + ArcFace on the cropped clips (via InsightFace) and clusters embeddings to assign a `signer_id` per clip. Videos with shift changes get per-clip ids; single-signer videos get one id shared by all their clips. Writes `data/cache/signers.json` and a preview contact sheet under `data/cache/signer_previews/`.

In [9]:
from identify_signers import identify_signers

signer_result = identify_signers(
    features_dir=FEATURES_DIR,
    output_path=SIGNERS_JSON,
    preview_dir=SIGNER_PREVIEWS,
)
print(f"\n{signer_result['n_clusters']} signers across {signer_result['videos']} videos")
if signer_result["multi_signer"]:
    print(f"Multi-signer videos: {signer_result['multi_signer']}")

Found 11 videos in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/features

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/xandro/.in

/opt/anaconda3/envs/usl/lib/python3.11/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  [OK]   2Nnz697BVTw: 24/24 frames
  [OK]   A2hCZVvtUSE: 24/24 frames
  [OK]   IOflFDS2biE: 24/24 frames
  [OK]   K9ouFMtz-s8: 24/24 frames
  [OK]   KUDt_SKkPUE: 24/24 frames
  [OK]   MSqpwfErl34: 24/24 frames
  [OK]   Nyykyn4FpNo: 24/24 frames
  [OK]   Q3yRVXmZdGQ: 24/24 frames
  [OK]   cNT6ajjEwVU: 24/24 frames
  [OK]   jj5jiyl2mh0: 23/24 frames
  [OK]   uGMgleLkjho: 24/24 frames

== Phase 1b: intra-video clustering to flag multi-signer videos ==
  [MULTI] IOflFDS2biE: sub-cluster sizes [18, 5, 1]
  [MULTI] Nyykyn4FpNo: sub-cluster sizes [15, 8, 1]

== Phase 2: per-clip pass for 2 flagged videos ==


/opt/anaconda3/envs/usl/lib/python3.11/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  IOflFDS2biE: 481/481 clips embedded (0 without detections)
  Nyykyn4FpNo: 215/215 clips embedded (0 without detections)

== Phase 3: global clustering ==
  samples=705 → raw n_clusters=11 (threshold=0.4)
  after temporal smoothing: n_clusters=5

Done.
  multi-signer videos: ['IOflFDS2biE', 'Nyykyn4FpNo']
  mapping: /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/signers.json
  preview: /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/signer_previews/signers.png

5 signers across 11 videos
Multi-signer videos: ['IOflFDS2biE', 'Nyykyn4FpNo']


## 7. Write Dataset Splits

Joins `splits.csv` (split assignment) + `annotations.csv` (text_norm) + `signers.json` (signer_id) and writes the publishable `data/usl-suspilne/{train,dev,test,test_unseen}.csv` with schema `name|text_norm|signer_id`.

In [10]:
from write_dataset_splits import write_dataset_splits

write_dataset_splits(
    splits_csv=SPLITS_CSV,
    annotations_csv=ANNOTATIONS_CSV,
    signers_path=SIGNERS_JSON,
    dataset_dir=CACHE_DIR,
)


  train: 1691 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/train.csv
  dev: 211 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/dev.csv
  test: 212 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/test.csv
  test_unseen: 0 clips -> /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/test_unseen.csv


{'counts': {'train': 1691, 'dev': 211, 'test': 212, 'test_unseen': 0}}

## 8. Extract Poses (MediaPipe Holistic)

Extracts 75 keypoints per frame: 33 body + 21 left hand + 21 right hand.
Output shape per clip: `(T, 225)` = 75 joints × (x, y, visibility).
Skips already-extracted clips.

In [11]:
from extract_poses import extract_all

extract_all(src=FEATURES_DIR, dst=HOLISTIC_DIR)

Found 2115 clips in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/features

  [1/2115] 2Nnz697BVTw/0000: 114 frames (cached)
  [2/2115] 2Nnz697BVTw/0001: 359 frames (cached)
  [3/2115] 2Nnz697BVTw/0002: 307 frames (cached)
  [4/2115] 2Nnz697BVTw/0003: 102 frames (cached)
  [5/2115] 2Nnz697BVTw/0004: 67 frames (cached)
  [6/2115] 2Nnz697BVTw/0005: 190 frames (cached)
  [7/2115] 2Nnz697BVTw/0006: 179 frames (cached)
  [8/2115] 2Nnz697BVTw/0007: 176 frames (cached)
  [9/2115] 2Nnz697BVTw/0008: 220 frames (cached)
  [10/2115] 2Nnz697BVTw/0009: 304 frames (cached)
  [11/2115] 2Nnz697BVTw/0010: 102 frames (cached)
  [12/2115] 2Nnz697BVTw/0011: 107 frames (cached)
  [13/2115] 2Nnz697BVTw/0012: 310 frames (cached)
  [14/2115] 2Nnz697BVTw/0013: 43 frames (cached)
  [15/2115] 2Nnz697BVTw/0014: 55 frames (cached)
  [16/2115] 2Nnz697BVTw/0015: 136 frames (cached)
  [17/2115] 2Nnz697BVTw/0016: 50 frames (cached)
  [18/2115] 2Nnz697BVTw/0017: 474 frames (cached)
  [19/2115] 2Nnz697BVTw/0018

{'total': 2115, 'failed': 0}

## 9. Lift 2D Holistic poses to 3D

Selects 50 joints from the 75-joint Holistic output and lifts them to 3D via the gopeith TF backprop optimizer. Output: `data/usl-suspilne/poses/mediapipe_3d/{videoId}/{clip}.npy` (T, 150).

In [12]:
from lift_to_3d import lift_all

lift_all(src=HOLISTIC_DIR, dst=MP3D_DIR)


Instructions for updating:
non-resource variables are not supported in the long term
Found 2115 Holistic pose files in /Users/xandro/code/diploma/ucu-text-to-sign/data/cache/poses/mediapipe_holistic

  [1/2115] 2Nnz697BVTw/0000: 114 frames (cached)
  [2/2115] 2Nnz697BVTw/0001: 359 frames (cached)
  [3/2115] 2Nnz697BVTw/0002: 307 frames (cached)
  [4/2115] 2Nnz697BVTw/0003: 102 frames (cached)
  [5/2115] 2Nnz697BVTw/0004: 67 frames (cached)
  [6/2115] 2Nnz697BVTw/0005: 190 frames (cached)
  [7/2115] 2Nnz697BVTw/0006: 179 frames (cached)
  [8/2115] 2Nnz697BVTw/0007: 176 frames (cached)
  [9/2115] 2Nnz697BVTw/0008: 220 frames (cached)
  [10/2115] 2Nnz697BVTw/0009: 304 frames (cached)
  [11/2115] 2Nnz697BVTw/0010: 102 frames (cached)
  [12/2115] 2Nnz697BVTw/0011: 107 frames (cached)
  [13/2115] 2Nnz697BVTw/0012: 310 frames (cached)
  [14/2115] 2Nnz697BVTw/0013: 43 frames (cached)
  [15/2115] 2Nnz697BVTw/0014: 55 frames (cached)
  [16/2115] 2Nnz697BVTw/0015: 136 frames (cached)
  [17/2115] 

{'total': 2115, 'skipped': 2115, 'failed': 0}

In [13]:
import csv
import subprocess
import json

# Count total clips on disk
total_clips = len(list(FEATURES_DIR.glob("*/*.mp4")))
print(f"Total clips: {total_clips}")
for vid_dir in sorted(FEATURES_DIR.iterdir()):
    if vid_dir.is_dir():
        clips = list(vid_dir.glob("*.mp4"))
        print(f"  {vid_dir.name}: {len(clips)} clips")

def _filter_csv(path):
    """Drop rows whose clip file doesn't exist on disk; rewrite in place."""
    with open(path) as f:
        rows = list(csv.DictReader(f, delimiter="|"))
    valid = []
    missing = []
    for row in rows:
        vid, clip = row["name"].split("/")
        if (FEATURES_DIR / vid / f"{clip}.mp4").exists():
            valid.append(row)
        else:
            missing.append(row["name"])
    if missing:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(valid[0].keys()), delimiter="|")
            writer.writeheader()
            writer.writerows(valid)
    return len(valid), missing

# Filter all dataset CSVs (plus the cache copy of annotations.csv for consistency)
for path in [ANNOTATIONS_CSV,
             CACHE_DIR / "train.csv",
             CACHE_DIR / "dev.csv",
             CACHE_DIR / "test.csv"]:
    n_valid, missing = _filter_csv(path)
    status = f"{n_valid} rows"
    if missing:
        by_vid = {}
        for name in missing:
            v = name.split("/")[0]
            by_vid[v] = by_vid.get(v, 0) + 1
        status += f" (removed {len(missing)}: {by_vid})"
    print(f"  {path.name}: {status}")

Total clips: 2115
  2Nnz697BVTw: 85 clips
  A2hCZVvtUSE: 115 clips
  IOflFDS2biE: 481 clips
  K9ouFMtz-s8: 61 clips
  KUDt_SKkPUE: 146 clips
  MSqpwfErl34: 118 clips
  Nyykyn4FpNo: 215 clips
  Q3yRVXmZdGQ: 145 clips
  cNT6ajjEwVU: 179 clips
  jj5jiyl2mh0: 423 clips
  uGMgleLkjho: 147 clips
  annotations.csv: 2114 rows
  train.csv: 1691 rows
  dev.csv: 211 rows
  test.csv: 212 rows


In [14]:
# Total duration
total_dur = 0
per_video = {}
for clip in sorted(FEATURES_DIR.glob("*/*.mp4")):
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(clip)],
        capture_output=True, text=True
    )
    dur = float(json.loads(result.stdout)["format"]["duration"])
    total_dur += dur
    vid = clip.parent.name
    per_video[vid] = per_video.get(vid, 0) + dur

for vid in sorted(per_video):
    m, s = divmod(per_video[vid], 60)
    print(f"  {vid}: {int(m)}m {s:.0f}s")

h, remainder = divmod(total_dur, 3600)
m, s = divmod(remainder, 60)
print(f"\nTotal duration: {int(h)}h {int(m)}m {s:.0f}s")

  2Nnz697BVTw: 12m 20s
  A2hCZVvtUSE: 15m 25s
  IOflFDS2biE: 36m 37s
  K9ouFMtz-s8: 7m 51s
  KUDt_SKkPUE: 16m 27s
  MSqpwfErl34: 10m 49s
  Nyykyn4FpNo: 19m 34s
  Q3yRVXmZdGQ: 11m 41s
  cNT6ajjEwVU: 22m 27s
  jj5jiyl2mh0: 37m 6s
  uGMgleLkjho: 13m 18s

Total duration: 3h 23m 35s


## 10. Refresh README Stats

Regenerates the `<!-- AUTO-STATS -->` block in `data/cache/README.md`. The
publish step below copies this README (along with features and poses) into
`data/usl-suspilne/`, so the published README always carries fresh numbers.

Prose elsewhere in the README is preserved.

In [15]:
import re
import numpy as np

README_PATH = CACHE_DIR / "README.md"
FPS = 30.0
README_SPLITS = ["train", "dev", "test"]


def _rows(split_name):
    with open(CACHE_DIR / f"{split_name}.csv", encoding="utf-8") as f:
        return list(csv.DictReader(f, delimiter="|"))


def _frames(name):
    vid, clip = name.split("/")
    p = HOLISTIC_DIR / vid / f"{clip}.npy"
    return np.load(p, mmap_mode="r").shape[0] if p.exists() else 0


stats = {}
for sp in README_SPLITS:
    rows = _rows(sp)
    frames = [_frames(r["name"]) for r in rows]
    tokens = [t for r in rows for t in r["text_norm"].split()]
    stats[sp] = {
        "clips": len(rows),
        "frames": sum(frames),
        "hours": sum(frames) / FPS / 3600,
        "videos": {r["name"].split("/")[0] for r in rows},
        "avg_s": (sum(frames) / FPS / len(rows)) if rows else 0.0,
        "signers": {r["signer_id"] for r in rows},
        "tokens": tokens,
    }

train_vocab = set(stats["train"]["tokens"])
total_clips = sum(s["clips"] for s in stats.values())
total_frames = sum(s["frames"] for s in stats.values())
total_hours = total_frames / FPS / 3600
all_videos = set().union(*(s["videos"] for s in stats.values()))
all_signers = set().union(*(s["signers"] for s in stats.values()))


def _oov_pct(toks):
    return 100 * sum(t not in train_vocab for t in toks) / len(toks) if toks else 0.0


lines = [
    "<!-- BEGIN AUTO-STATS -->",
    "<!-- Regenerated by build_dataset.ipynb. Do not edit by hand. -->",
    "",
    "## Statistics",
    "",
    "*Auto-generated from the dataset on disk.*",
    "",
    "| split | clips | hours | videos | avg s/clip |",
    "|-------|-------|-------|--------|------------|",
]
for sp in README_SPLITS:
    s = stats[sp]
    lines.append(
        f"| {sp} | {s['clips']} | {s['hours']:.2f} | {len(s['videos'])} | {s['avg_s']:.1f} |"
    )
avg_total = total_frames / FPS / total_clips if total_clips else 0.0
lines.append(
    f"| total | {total_clips} | {total_hours:.2f} | {len(all_videos)} | {avg_total:.1f} |"
)
lines += [
    "",
    f"- Total frames: {total_frames:,}",
    f"- Signers: {len(all_signers)}",
    f"- Train vocabulary: {len(train_vocab):,} types / {len(stats['train']['tokens']):,} tokens",
    f"- OOV (token-level): dev {_oov_pct(stats['dev']['tokens']):.1f}%,"
    f" test {_oov_pct(stats['test']['tokens']):.1f}%",
    "",
    "<!-- END AUTO-STATS -->",
]
new_block = "\n".join(lines)

text = README_PATH.read_text(encoding="utf-8")
text = re.sub(
    r"<!-- BEGIN AUTO-STATS -->.*?<!-- END AUTO-STATS -->",
    lambda _m: new_block,
    text,
    flags=re.DOTALL,
)
README_PATH.write_text(text, encoding="utf-8")
print(f"Updated {README_PATH.relative_to(ROOT)}")


Updated data/cache/README.md


## 11. Publish to `data/usl-suspilne/` with v-IDs

Hardlinks (or copies, if cache and dataset dirs sit on different filesystems)
`cache/{features,poses}/{yt_id}/` to `usl-suspilne/{features,poses}/{v_id}/`,
writes split CSVs with v-IDs in the `name` column, and copies `cache/README.md`
to the published tree.

Hardlinks mean the published tree shares storage with the cache — no 2× disk
cost. Idempotent: re-runs only link new files. Each video keeps the same v-ID
across rebuilds (mapping persists in `data/cache/videos.json`).

In [16]:
from publish_dataset import publish_dataset

result = publish_dataset(
    cache_dir=CACHE_DIR,
    dataset_dir=DATASET_DIR,
    videos_json=VIDEOS_JSON,
    splits=("train", "dev", "test"),
)

print(f"Discovered videos: {result['discovered_videos']}")
print(f"Files linked:      {result['files_added']}")
print(f"CSVs written:      {result['csvs_written']} ({result['rows_published']} rows)")
print()
print("YT_ID → v_ID:")
for yt_id, v_id in sorted(result["mapping"].items(), key=lambda kv: kv[1]):
    print(f"  {v_id}  ←  {yt_id}")


Discovered videos: 11
Files linked:      0
CSVs written:      3 (2114 rows)

YT_ID → v_ID:
  v00  ←  2Nnz697BVTw
  v01  ←  A2hCZVvtUSE
  v02  ←  IOflFDS2biE
  v03  ←  K9ouFMtz-s8
  v04  ←  KUDt_SKkPUE
  v05  ←  MSqpwfErl34
  v06  ←  Nyykyn4FpNo
  v07  ←  Q3yRVXmZdGQ
  v08  ←  cNT6ajjEwVU
  v09  ←  jj5jiyl2mh0
  v10  ←  uGMgleLkjho
